# 07 · Plate reader: base

The YOLO26s plate reader trained with exactly the steps of notebook 05 (150 epochs, 320 px, batch 64, no mosaic,
no left-right flip). It is the baseline for notebooks 08 to 13, which each add one change.
All of 07 to 13 must use the same version of the `bangla-plate-reading` dataset.

In [ ]:
%pip install -q ultralytics==8.4.46

## Configuration

In [ ]:
EXPERIMENT = "yolo26s-reader"

EXPERIMENTS = {
    "yolo26n-reader": {"model": "yolo26n.pt", "batch": 64},
    "yolo26s-reader": {"model": "yolo26s.pt", "batch": 64},
}
EPOCHS = 150
IMGSZ = 320
SEED = 0

## Setup

In [ ]:
import json
import shutil
from pathlib import Path

import pandas as pd
import torch
from ultralytics import YOLO

config = EXPERIMENTS[EXPERIMENT]
WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
RUNS = WORK / "runs"

gpus = torch.cuda.device_count()
DEVICE = ",".join(str(i) for i in range(gpus)) if gpus else "cpu"

# copy the dataset so Ultralytics can write its label cache
DATA = WORK / "plate-reading"
if not DATA.exists():
    shutil.copytree(next(p for p in INPUT.rglob("bangla-plate-reading") if p.is_dir()), DATA)
DATA_YAML = DATA / "data.yaml"
RUN = RUNS / EXPERIMENT
WEIGHTS = RUN / "weights" / "best.pt"
CLASSES = json.loads((DATA / "classes.json").read_text(encoding="utf-8"))

print(f"Experiment: {EXPERIMENT}  {config}\nDevice: {DEVICE}")
for split in ("train", "val", "test"):
    print(f"{split:5} {len(list((DATA / 'images' / split).iterdir())):6d} crops")

## Train

In [ ]:
YOLO(config["model"]).train(
    data=str(DATA_YAML), imgsz=IMGSZ, epochs=EPOCHS, batch=config["batch"], patience=0,
    fliplr=0.0, mosaic=0.0,
    device=DEVICE, seed=SEED, project=str(RUNS), name=EXPERIMENT, exist_ok=True,
)
print("Weights:", WEIGHTS)

## Training curves

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv(RUN / "results.csv")
df.columns = df.columns.str.strip()
best = df.loc[df["metrics/mAP50-95(B)"].idxmax()]
print(f"Epochs: {len(df)}  best epoch {int(best['epoch'])}: "
      f"mAP@0.5 {best['metrics/mAP50(B)']:.4f}  mAP@0.5:0.95 {best['metrics/mAP50-95(B)']:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for keys, ax, title in [
    (("train/box_loss", "val/box_loss"), axes[0], "Box loss"),
    (("train/cls_loss", "val/cls_loss"), axes[1], "Class loss"),
    (("metrics/mAP50(B)", "metrics/mAP50-95(B)"), axes[2], "mAP"),
]:
    for key in keys:
        ax.plot(df["epoch"], df[key], label=key)
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.show()

## Test: characters

mAP on the test split, per class and by class kind.

In [ ]:
model = YOLO(WEIGHTS)
metrics = model.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=config["batch"], device=DEVICE,
                    project=str(RUNS), name=f"{EXPERIMENT}-test", exist_ok=True, plots=False)
print(f"Test  P {metrics.box.mp:.4f}  R {metrics.box.mr:.4f}  "
      f"mAP@0.5 {metrics.box.map50:.4f}  mAP@0.5:0.95 {metrics.box.map:.4f}")

per_class = pd.DataFrame({
    "class": [CLASSES[i]["name"] for i in metrics.box.ap_class_index],
    "kind": [CLASSES[i]["kind"] for i in metrics.box.ap_class_index],
    "mAP50": metrics.box.ap50,
    "mAP50-95": metrics.box.ap,
})
print(per_class.groupby("kind")[["mAP50", "mAP50-95"]].mean().round(4))
print("\nWeakest classes:")
print(per_class.sort_values("mAP50").head(10).round(4).to_string(index=False))

## Test: plate text

The detections of each crop are turned into plate text: boxes are split into a top and a bottom line at the largest vertical gap, each line is read left to right, and six digits are written as `12-3456`. The labels go through the same function, so the comparison does not depend on formatting.

- **Plate accuracy**: share of plates whose full text is correct.
- **Character error rate (CER)**: edit distance between predicted and true text, divided by the length of the true text.

In [ ]:
BANGLA = [c["bangla"] for c in CLASSES]
KIND = [c["kind"] for c in CLASSES]


def plate_text(boxes):
    # boxes: (class, x_center, y_center, width, height), normalised
    if not boxes:
        return ""
    boxes = sorted(boxes, key=lambda b: b[2])
    lines = [boxes]
    if len(boxes) > 1:
        gaps = [boxes[i + 1][2] - boxes[i][2] for i in range(len(boxes) - 1)]
        split = max(range(len(gaps)), key=gaps.__getitem__)
        median_h = sorted(b[4] for b in boxes)[len(boxes) // 2]
        if gaps[split] > 0.5 * median_h:
            lines = [boxes[:split + 1], boxes[split + 1:]]
    out = []
    for line in lines:
        line = sorted(line, key=lambda b: b[1])
        tokens = [BANGLA[int(b[0])] for b in line]
        if all(KIND[int(b[0])] == "digit" for b in line):
            digits = "".join(tokens)
            out.append(f"{digits[:2]}-{digits[2:]}" if len(digits) == 6 else digits)
        else:
            # keep runs of digits together: ["ঢাকা", "১", "২"] -> "ঢাকা ১২"
            words = []
            for b, token in zip(line, tokens):
                if words and KIND[int(b[0])] == "digit" and words[-1][0] == "digit":
                    words[-1][1] += token
                else:
                    words.append([KIND[int(b[0])], token])
            out.append(" ".join(w for _, w in words))
    return " ".join(out)


def edit_distance(a, b):
    row = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j, cb in enumerate(b, 1):
            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (ca != cb))
    return row[-1]


model = YOLO(WEIGHTS)
images = sorted((DATA / "images" / "test").iterdir())
rows = []
for start in range(0, len(images), 64):
    batch = images[start:start + 64]
    for path, result in zip(batch, model.predict(batch, imgsz=IMGSZ, device=DEVICE, agnostic_nms=True, verbose=False)):
        predicted = [(int(c), *xywh) for c, xywh in zip(result.boxes.cls.tolist(), result.boxes.xywhn.tolist())]
        label = DATA / "labels" / "test" / f"{path.stem}.txt"
        truth = [(int(p[0]), *map(float, p[1:])) for p in (l.split() for l in label.read_text().splitlines()) if p]
        t, p = plate_text(truth), plate_text(predicted)
        rows.append({"image": path.name, "source": path.name.split("__")[0], "truth": t, "predicted": p,
                     "correct": t == p, "errors": edit_distance(t, p), "length": len(t)})

plates = pd.DataFrame(rows)
for name, group in [("all", plates)] + list(plates.groupby("source")):
    print(f"{name:7} plates {len(group):4d}  plate accuracy {group['correct'].mean():.2%}  "
          f"CER {group['errors'].sum() / group['length'].sum():.2%}")
plates.to_csv(RUN / "test_plates.csv", index=False)
print("\nExamples of wrong plates:")
print(plates[~plates["correct"]].head(15)[["truth", "predicted"]].to_string(index=False))

## Save

In [ ]:
shutil.copy2(DATA / "classes.json", RUN / "weights" / "classes.json")
shutil.make_archive(str(WORK / EXPERIMENT), "zip", RUN)
print("Saved", WORK / f"{EXPERIMENT}.zip")